In [ ]:
import pandas as pd
import numpy as np
from IPython.display import FileLink
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb

# Load data
print("Loading data...")
train_df = pd.read_csv('/kaggle/input/iml-fall-2024-challenge-1/train_set.csv')
test_df = pd.read_csv('/kaggle/input/iml-fall-2024-challenge-1/test_set.csv')
print("Data loaded successfully.")

# Impute missing values
print("Imputing missing values...")
imputer = SimpleImputer(strategy='mean')
train_df.loc[:, train_df.columns != 'Y'] = imputer.fit_transform(train_df.loc[:, train_df.columns != 'Y'])
print("Missing values imputed.")

# Correlation-based feature reduction
print("Performing correlation-based feature reduction...")
correlation_matrix = train_df.corr().abs()
correlation_threshold = 0.8
upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > correlation_threshold)]
train_df_reduced = train_df.drop(columns=to_drop)
print(f"Dropped {len(to_drop)} highly correlated features.")

Features_reduced = train_df_reduced.drop(columns=['RecordId', 'Y'])
Target = train_df_reduced['Y']

# Scaling
print("Scaling features using RobustScaler...")
scaler = RobustScaler()
Features = scaler.fit_transform(Features_reduced)
print("Features scaled.")

# Train-test split
print("Splitting data into train and validation sets...")
train_f, val_f, train_t, val_t = train_test_split(Features, Target, test_size=0.30, random_state=42)
train_data = xgb.DMatrix(train_f, label=train_t)
val_data = xgb.DMatrix(val_f, label=val_t)
print("Train-test split completed.")

# Define XGBoost parameters
base_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.4,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.5,
    'colsample_bynode': 0.6,
    'scale_pos_weight': 8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'alpha': 3,
    'lambda': 3,
    'max_delta_step': 2,
    'tree_method': 'hist',
    'verbosity': 1
}

# Train multiple models with varying parameters
print("Training multiple models...")
num_models = 200
num_rounds = 500
models = []
weights = []

for i in range(num_models):
    if i % 10 == 0:  # Print progress every 10 models
        print(f"Training model {i + 1}/{num_models}...")
    
    params = base_params.copy()
    params['seed'] = i * 10
    params['eta'] = 0.05 + i * 0.001
    params['colsample_bytree'] = 0.8 - i * 0.001
    
    model = xgb.train(params, train_data, num_boost_round=num_rounds, verbose_eval=False)
    models.append(model)
    
    pred_prob = model.predict(val_data)
    auc_score = roc_auc_score(val_t, pred_prob)
    weights.append(auc_score)

# Normalize weights
weights = np.array(weights)
weights /= weights.sum()
print("Weights normalized.")

# Weighted predictions on validation set
print("Generating weighted predictions on the validation set...")
val_preds_weighted = np.zeros(len(val_t))
for model, weight in zip(models, weights):
    pred_prob = model.predict(val_data)
    val_preds_weighted += weight * pred_prob

val_predictions = [1 if prob > 0.5 else 0 for prob in val_preds_weighted]
accuracy = accuracy_score(val_t, val_predictions)
print(f"Validation Accuracy: {accuracy}")

# Prepare test data
print("Preparing test data...")
test_df_imputed = imputer.transform(test_df)
test_features = pd.DataFrame(test_df_imputed, columns=test_df.columns)
test_features = test_features.drop(columns=['RecordId'] + to_drop)
test_data = xgb.DMatrix(test_features)
print("Test data prepared.")

# Weighted predictions on test set
print("Generating weighted predictions on the test set...")
test_preds_weighted = np.zeros(len(test_features))
for model, weight in zip(models, weights):
    test_pred_prob = model.predict(test_data)
    test_preds_weighted += weight * test_pred_prob

# Create submission DataFrame
print("Creating submission file...")
submission = pd.DataFrame({'RecordId': test_df['RecordId'], 'Y': test_preds_weighted})
submission_file = 'submission.csv'
submission.to_csv(submission_file, index=False)
print("Submission file created successfully!")

# Provide download link
display(FileLink(submission_file))


Loading data...
Data loaded successfully.
Imputing missing values...
Missing values imputed.
Performing correlation-based feature reduction...
Dropped 21 highly correlated features.
Scaling features using RobustScaler...
Features scaled.
Splitting data into train and validation sets...
Train-test split completed.
Training multiple models...
Training model 1/200...
Training model 11/200...
Training model 21/200...
Training model 31/200...
